In [2]:
import os
from google.colab import drive

# 1. 구글 드라이브 마운트
drive.mount('/content/drive')
save_path = '/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021'
os.makedirs(save_path, exist_ok=True)

# 2. 라이브러리 설치
!pip install -q cellxgene_census scanpy

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/202.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 22.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.


In [ ]:
import cellxgene_census

# Ren et al. (2021) Dataset ID
# ID: 9dbab10c-118d-496b-966a-67f1763a6b7d
dataset_id = "9dbab10c-118d-496b-966a-67f1763a6b7d"
file_name = "Ren_et_al_COVID_2021.h5ad"
full_path = os.path.join(save_path, file_name)

print(f"Target Path: {full_path}")
print("Starting Stream Download (Bypassing RAM)...")

try:
    cellxgene_census.download_source_h5ad(
        dataset_id,
        to_path=full_path
    )
    print("✅ Download Complete! The file is safely in your Drive.")

except Exception as e:
    print(f"❌ Error: {e}")
    print("혹시 이미 파일이 존재하거나 경로 문제가 있는지 확인해주세요.")

# 연결 끊김 방지용 더미 코드 (선택)
import time
for i in range(10):
    time.sleep(60)
    print(f"Keeping session alive... {i+1} min")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Target Path: /content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/Ren_et_al_COVID_2021.h5ad
Starting Stream Download (Bypassing RAM)...


Downloading: 100%|██████████| 13.0G/13.0G [18:51<00:00, 12.4MB/s]


✅ Download Complete! The file is safely in your Drive.
Keeping session alive... 1 min
Keeping session alive... 2 min
Keeping session alive... 3 min
Keeping session alive... 4 min
Keeping session alive... 5 min
Keeping session alive... 6 min
Keeping session alive... 7 min
Keeping session alive... 8 min
Keeping session alive... 9 min
Keeping session alive... 10 min


In [3]:
import scanpy as sc
import pandas as pd
import numpy as np
import gc

# 경로 설정 (아까 다운받은 파일)
path = '/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/Ren_et_al_COVID_2021.h5ad'


print("1. Loading Data (Backed Mode)...")
# backed='r'로 메타데이터만 먼저 읽어서 메모리 절약
adata = sc.read_h5ad(input_path, backed='r')
print(f"Original Shape: {adata.shape}")

1. Loading Data (Backed Mode)...
Original Shape: (1462702, 27131)


In [5]:
obs = adata.obs
print(obs.columns)

print("=== [CoVID-19 severity] Unique Values ===")
print(adata.obs['CoVID-19 severity'].unique())

print("\n=== [disease] Unique Values ===")
print(adata.obs['disease'].unique())

Index(['celltype', 'majorType', 'City', 'sampleID', 'donor_id', 'Sample type',
       'CoVID-19 severity', 'Sample time',
       'Sampling day (Days after symptom onset)', 'BCR single cell sequencing',
       'TCR single cell sequencing', 'Outcome', 'Comorbidities',
       'COVID-19-related medication and anti-microbials',
       'Leukocytes [G over L]', 'Neutrophils [G over L]',
       'Lymphocytes [G over L]', 'Unpublished', 'disease_ontology_term_id',
       'cell_type_ontology_term_id', 'tissue_ontology_term_id',
       'development_stage_ontology_term_id',
       'self_reported_ethnicity_ontology_term_id', 'assay_ontology_term_id',
       'sex_ontology_term_id', 'is_primary_data', 'suspension_type',
       'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue',
       'self_reported_ethnicity', 'development_stage', 'observation_joinid'],
      dtype='object')
=== [CoVID-19 severity] Unique Values ===
['severe/critical', 'mild/moderate', 'control']
Categories (3, object): 

In [6]:
# ---------------------------------------------------------
# [Step 1] 타겟 라벨 매핑 및 필터링
# ---------------------------------------------------------
print("🎯 2. Filtering Target Cells based on [CoVID-19 severity]...")

severity_col = 'CoVID-19 severity'

# MIL 모델용 Ordinal Mapping (0 -> 1 -> 2)
label_map = {
    'control': 0,          # Healthy / Normal
    'mild/moderate': 1,    # Early / Intermediate Disease
    'severe/critical': 2   # Advanced Disease
}

# 타겟 라벨에 해당하는 세포만 선택 (여기서는 사실상 전체 데이터)
mask = adata.obs[severity_col].isin(label_map.keys())
print(f"   Selected {sum(mask)} cells out of {adata.shape[0]}")

🎯 2. Filtering Target Cells based on [CoVID-19 severity]...
   Selected 1462702 cells out of 1462702


In [9]:
import pandas as pd
import numpy as np

OBS = adata.obs.copy()

BAG = "sampleID"
DONOR = "donor_id"
SEV = "CoVID-19 severity"

# --- basic sanity: required cols exist ---
for c in [BAG, DONOR, SEV]:
    assert c in OBS.columns, f"missing column: {c}"

# --- drop missing key fields ---
before = len(OBS)
OBS = OBS.dropna(subset=[BAG, DONOR, SEV])
print("dropped rows (missing key fields):", before - len(OBS))

# force string ids
OBS[BAG] = OBS[BAG].astype(str)
OBS[DONOR] = OBS[DONOR].astype(str)
OBS[SEV] = OBS[SEV].astype(str)

# --- 1) sample -> donor single-valued ---
s2d_nuniq = OBS.groupby(BAG)[DONOR].nunique()
bad_s2d = s2d_nuniq[s2d_nuniq > 1]
print("bad sample->donor (n>1):", len(bad_s2d))
if len(bad_s2d) > 0:
    print(bad_s2d.head(10))

# --- 2) sample -> severity single-valued ---
s2sev_nuniq = OBS.groupby(BAG)[SEV].nunique()
bad_s2sev = s2sev_nuniq[s2sev_nuniq > 1]
print("bad sample->severity (n>1):", len(bad_s2sev))
if len(bad_s2sev) > 0:
    # show which severities inside first few bad samples
    ex = bad_s2sev.index[:5]
    for sid in ex:
        vals = OBS.loc[OBS[BAG]==sid, SEV].value_counts()
        print("\nSample", sid, "severity counts:\n", vals)

# --- 3) build bag table (one row per sampleID) ---
bag_tbl = (
    OBS.groupby(BAG)
       .agg(
            donor_id=(DONOR, lambda x: x.iloc[0]),
            severity=(SEV, lambda x: x.iloc[0]),
            n_cells=(SEV, "size"),
       )
       .reset_index()
)

print("n_bags:", len(bag_tbl), "n_donors:", bag_tbl["donor_id"].nunique())
print("\nBags per severity:\n", bag_tbl["severity"].value_counts())
print("\nCells per bag (quantiles):\n", bag_tbl["n_cells"].quantile([0, .01, .05, .1, .25, .5, .75, .9, .95, .99, 1.0]))


dropped rows (missing key fields): 0
bad sample->donor (n>1): 0
bad sample->severity (n>1): 0
n_bags: 284 n_donors: 196

Bags per severity:
 severity
severe/critical    134
mild/moderate      122
control             28
Name: count, dtype: int64

Cells per bag (quantiles):
 0.00       57.00
0.01      159.15
0.05      309.90
0.10      554.50
0.25     2583.50
0.50     4970.50
0.75     6921.50
0.90     9813.60
0.95    11777.00
0.99    15823.73
1.00    18236.00
Name: n_cells, dtype: float64


In [10]:
sev_map = {
    "control": 0,
    "mild/moderate": 1,
    "severe/critical": 2
}

# normalize strings
bag_tbl["severity_norm"] = bag_tbl["severity"].str.strip().str.lower()
# dataset마다 표기 흔들릴 수 있어 대응
bag_tbl["y"] = bag_tbl["severity_norm"].map(sev_map)

missing = bag_tbl[bag_tbl["y"].isna()]["severity"].value_counts()
print("unmapped severity values:\n", missing)

# ordinal targets
bag_tbl["t1_sick"] = (bag_tbl["y"] > 0).astype(int)   # y>0
bag_tbl["t2_sev"]  = (bag_tbl["y"] > 1).astype(int)   # y>1

print(bag_tbl[["severity","y","t1_sick","t2_sev"]].head())

unmapped severity values:
 Series([], Name: count, dtype: int64)
  severity  y  t1_sick  t2_sev
0  control  0        0       0
1  control  0        0       0
2  control  0        0       0
3  control  0        0       0
4  control  0        0       0


In [16]:
import numpy as np
import pandas as pd

# -----------------------
# Config (FIX THESE)
# -----------------------
BAG = "sampleID"
DONOR = "donor_id"
SEV = "CoVID-19 severity"

CAP = 4096
MIN_CELLS = 100          # <- 네가 원하는 값으로 고정 (권장: 100~200)
SEED = 0                 # <- 재현성 seed 고정

# -----------------------
# Build bag table
# -----------------------
obs = adata.obs.copy()

# basic sanitation
req = [BAG, DONOR, SEV]
for c in req:
    assert c in obs.columns, f"missing column: {c}"
obs = obs.dropna(subset=req)

obs[BAG] = obs[BAG].astype(str)
obs[DONOR] = obs[DONOR].astype(str)
obs[SEV] = obs[SEV].astype(str)

# bag sizes
bag_sizes = obs.groupby(BAG).size().rename("n_cells").to_frame()

# keep bags with enough cells
keep_bags = bag_sizes.index[bag_sizes["n_cells"] >= MIN_CELLS]
print(f"[bags] total={bag_sizes.shape[0]} kept(MIN>={MIN_CELLS})={len(keep_bags)} dropped={bag_sizes.shape[0]-len(keep_bags)}")

# restrict obs to kept bags
obs_k = obs.loc[obs[BAG].isin(keep_bags)].copy()

# -----------------------
# Sample cells per bag up to CAP (fixed once)
# -----------------------
rng = np.random.default_rng(SEED)

# pre-get integer positions for fast slicing
# NOTE: adata.obs.index are cell IDs; we will return cell IDs and integer positions
cell_ids = obs_k.index.to_numpy()

# map cell_id -> integer position in adata (needed for adata[mask])
# safest: use get_indexer against adata.obs_names
pos_in_adata = adata.obs_names.get_indexer(cell_ids)
assert np.all(pos_in_adata >= 0), "some sampled cells not found in adata.obs_names (index mismatch?)"

# group integer positions by bag
tmp = pd.DataFrame({
    "cell_id": cell_ids,
    "pos": pos_in_adata,
    "bag": obs_k[BAG].to_numpy(),
})

selected_pos = []
selected_cell_ids = []

# sample within each bag
for bag, g in tmp.groupby("bag", sort=False):
    pos = g["pos"].to_numpy()
    if len(pos) <= CAP:
        pick = pos
        # keep same order doesn't matter; but for reproducibility, sort
        pick = np.sort(pick)
    else:
        pick = rng.choice(pos, size=CAP, replace=False)
        pick = np.sort(pick)
    selected_pos.append(pick)

selected_pos = np.concatenate(selected_pos, axis=0)
# optional: ensure unique and sorted global
selected_pos = np.unique(selected_pos)
print(f"[cells] selected={len(selected_pos)} (cap={CAP})")

# -----------------------
# Optional: materialize filtered AnnData (can be big)
# -----------------------
# WARNING: this can be memory heavy depending on X/obsm etc.
# Prefer backed='r' workflows or save indices first.

#adata_cap = adata[selected_pos].copy()
#print(adata_cap)

# -----------------------
# Save indices for reproducibility (recommended)
# -----------------------
np.save("/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/ren_cap4096_min100_seed0_cellpos.npy", selected_pos)
pd.Series(keep_bags).to_csv("/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/ren_keptbags_min100.txt", index=False, header=False)


[bags] total=284 kept(MIN>=100)=282 dropped=2
[cells] selected=914746 (cap=4096)


In [18]:
bag_tbl = (
    adata.obs
    .groupby("sampleID")
    .agg(
        severity=("CoVID-19 severity", "first"),
        n_cells=("CoVID-19 severity", "size")
    )
)

print(bag_tbl.groupby("severity")["n_cells"].describe())


                 count         mean          std     min      25%     50%  \
severity                                                                    
control           28.0  5892.857143  2574.816523  2436.0  3958.00  5086.5   
mild/moderate    122.0  5745.639344  3215.291594   274.0  3578.75  5696.5   
severe/critical  134.0  4453.238806  3815.862378    57.0  1211.50  3749.0   

                     75%      max  
severity                           
control          7403.75  10772.0  
mild/moderate    7587.25  15964.0  
severe/critical  6296.25  18236.0  


/tmp/ipython-input-1545038244.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("sampleID")
/tmp/ipython-input-1545038244.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(bag_tbl.groupby("severity")["n_cells"].describe())


In [ ]:
import os

OUT_H5AD = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100.h5ad"
os.makedirs(os.path.dirname(OUT_H5AD), exist_ok=True)

adata_cap_view = adata[selected_pos]          # view (no .copy)
adata_cap_view.write_h5ad(OUT_H5AD)           # write to new file
print("saved:", OUT_H5AD)

In [1]:
SRC_H5AD = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/Ren_et_al_COVID_2021.h5ad"
import h5py
with h5py.File(SRC_H5AD, "r") as f:
    print("X encoding-type:", f["X"].attrs.get("encoding-type"))
    print("X keys:", list(f["X"].keys())[:10])
    print("obs sampleID type:", type(f["obs"]["sampleID"]))


X encoding-type: csr_matrix
X keys: ['data', 'indices', 'indptr']
obs sampleID type: <class 'h5py._hl.group.Group'>


In [3]:
import h5py, numpy as np

SRC_H5AD = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/Ren_et_al_COVID_2021.h5ad"

with h5py.File(SRC_H5AD, "r") as f:
    # 1) raw 그룹 존재 여부
    print("has /raw:", "raw" in f.keys())
    if "raw" in f:
        print("raw keys:", list(f["raw"].keys()))
        if "X" in f["raw"]:
            print("raw/X encoding:", f["raw"]["X"].attrs.get("encoding-type"))
            print("raw/X keys:", list(f["raw"]["X"].keys()))

    # 2) X의 data dtype / 값 범위 샘플링
    X = f["X"]
    d = X["data"]
    print("X/data dtype:", d.dtype)

    # 앞부분 일부만 샘플링
    m = min(200000, d.shape[0])
    samp = d[:m]
    print("sample min/max:", float(samp.min()), float(samp.max()))
    # 정수성 체크 (float이면 소수 존재 여부)
    if np.issubdtype(samp.dtype, np.floating):
        frac = np.abs(samp - np.round(samp)).mean()
        print("avg |x-round(x)|:", float(frac))


has /raw: True
raw keys: ['X', 'var', 'varm']
raw/X encoding: csr_matrix
raw/X keys: ['data', 'indices', 'indptr']
X/data dtype: float32
sample min/max: 0.39759454131126404 8.179561614990234
avg |x-round(x)|: 0.28273260593414307


In [2]:
import h5py

with h5py.File(SRC_H5AD, "r") as f:
    print("var keys:", list(f["var"].keys()))
    if "raw" in f and "var" in f["raw"]:
        print("raw/var keys:", list(f["raw"]["var"].keys()))


var keys: ['_index', 'feature_biotype', 'feature_is_filtered', 'feature_length', 'feature_name', 'feature_reference', 'feature_type']
raw/var keys: ['_index', 'feature_biotype', 'feature_length', 'feature_name', 'feature_reference', 'feature_type']


In [3]:
import h5py, numpy as np, re

with h5py.File(SRC_H5AD, "r") as f:
    v = f["var"]["_index"][()]
    # bytes -> str
    if v.dtype.kind in ("S","O"):
        v = v.astype("U")
    s = v[:2000]

ens = np.mean([bool(re.match(r"^ENS[A-Z]*G\d{5,}", x)) for x in s])
print("fraction Ensembl-like:", ens)

# 중복 여부도 체크(중복이면 symbol일 확률↑ 또는 처리 필요)
uniq = len(np.unique(s))
print("unique in sample:", uniq, "/", len(s))


fraction Ensembl-like: 1.0
unique in sample: 2000 / 2000


In [11]:
import os
import time
import shutil
import numpy as np
import h5py

# -----------------------
# CONFIG
# -----------------------
SRC_H5AD = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/Ren_et_al_COVID_2021.h5ad"
OUT_DIR_DRIVE = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021"
LOCAL_DST = "/content/adata_cap4096_min100_milmin_rawcounts.h5ad"
DRIVE_DST = os.path.join(OUT_DIR_DRIVE, "adata_cap4096_min100_milmin_rawcounts.h5ad")
SEL_NPY = os.path.join(OUT_DIR_DRIVE, "ren_cap4096_min100_seed0_cellpos.npy")

KEEP_OBS_COLS = ["sampleID", "donor_id", "CoVID-19 severity", "celltype", "majorType", "City"]
KEEP_RAW_VAR_COLS = ["feature_biotype", "feature_length", "feature_name",
                     "feature_reference", "feature_type"]

CSR_CHUNK_ROWS = 32768

# -----------------------
# LOAD SELECTION
# -----------------------
selected_pos = np.load(SEL_NPY).astype(np.int64)
selected_pos.sort()
n_sel = len(selected_pos)

print("[sel] n_cells:", n_sel)

# -----------------------
# HELPERS
# -----------------------

def _as_str(arr):
    if arr.dtype.kind == "U":
        return arr
    if arr.dtype.kind in ("S", "O"):
        return arr.astype("U")
    return arr

def iter_runs(sorted_idx):
    s = sorted_idx
    if len(s) == 0:
        return
    start = s[0]
    prev = s[0]
    for x in s[1:]:
        if x == prev + 1:
            prev = x
        else:
            yield start, prev + 1
            start = x
            prev = x
    yield start, prev + 1

def read_ds_by_runs(ds, sorted_idx):
    parts = []
    for a, b in iter_runs(sorted_idx):
        parts.append(ds[a:b])
    if not parts:
        return ds[:0]
    out = np.concatenate(parts, axis=0)
    if out.dtype.kind in ("S", "O"):
        out = _as_str(out)
    return out

def decode_categorical_group(g, row_indexer):
    cats = _as_str(g["categories"][()])
    codes = read_ds_by_runs(g["codes"], row_indexer)

    out = np.empty(len(codes), dtype=object)  # <-- 핵심: object로
    for i, c in enumerate(codes):
        out[i] = "" if c < 0 else str(cats[int(c)])
    return out

def read_obs_col(obs_group, col, row_indexer):
    obj = obs_group[col]
    if isinstance(obj, h5py.Dataset):
        return read_ds_by_runs(obj, row_indexer)
    if isinstance(obj, h5py.Group):
        return decode_categorical_group(obj, row_indexer)
    raise RuntimeError(f"Unknown obs type: {type(obj)}")

# NEW: var column reader (Dataset vs categorical Group)
def read_var_col(var_group, col):
    obj = var_group[col]

    # Case 1: Dataset
    if isinstance(obj, h5py.Dataset):
        arr = obj[()]
        if arr.dtype.kind in ("S", "O"):
            arr = arr.astype("U")
        return arr

    # Case 2: Group (Categorical)
    if isinstance(obj, h5py.Group):
        if "categories" in obj and "codes" in obj:
            cats = _as_str(obj["categories"][()])
            codes = obj["codes"][()]
            # (Note: Vectorization logic below is better, but keeping flow for structure)
            out = np.empty(len(codes), dtype=object)
            for i, code in enumerate(codes):
                out[i] = "" if code < 0 else str(cats[int(code)])
            return out

        raise RuntimeError(f"[var] unsupported group encoding for {col}: keys={list(obj.keys())}")

    raise RuntimeError(f"[var] unknown type for {col}: {type(obj)}")

def write_dataframe_group(fout, key, index_vals, cols_dict):
    g = fout.create_group(key)
    g.attrs["encoding-type"] = "dataframe"
    g.attrs["encoding-version"] = "0.2.0"

    str_dt = h5py.string_dtype("utf-8")

    # AnnData expects this attribute to exist on the dataframe group
    # and to contain the name of the dataset holding the index values.
    g.attrs["_index"] = "_index"

    # index dataset
    g.create_dataset("_index", data=_as_str(index_vals).astype(object), dtype=str_dt)

    col_order = []
    for c, arr in cols_dict.items():
        col_order.append(c)
        a = np.asarray(arr)
        if a.dtype.kind in ("U", "S", "O"):
            g.create_dataset(c, data=_as_str(a).astype(object), dtype=str_dt)
        else:
            g.create_dataset(c, data=a, dtype=a.dtype)

    g.attrs["column-order"] = np.asarray(col_order).astype(object)

def csr_subset(Xin, fout, selected_rows, n_vars):
    data_in = Xin["data"]
    ind_in  = Xin["indices"]
    ptr_in  = Xin["indptr"]

    indptr = ptr_in[()]
    n_out = len(selected_rows)

    Xout = fout.create_group("X")
    Xout.attrs["encoding-type"] = "csr_matrix"
    Xout.attrs["encoding-version"] = "0.1.0"
    Xout.attrs["shape"] = np.array([n_out, n_vars], dtype=np.int64)

    d_data = Xout.create_dataset("data", shape=(0,), maxshape=(None,),
                                 dtype=data_in.dtype, chunks=True)
    d_ind  = Xout.create_dataset("indices", shape=(0,), maxshape=(None,),
                                 dtype=ind_in.dtype, chunks=True)
    d_ptr  = Xout.create_dataset("indptr", shape=(n_out + 1,),
                                 dtype=ptr_in.dtype)

    d_ptr[0] = 0
    nnz_total = 0
    out_row = 0

    print("[csr] start")

    for chunk_i, start in enumerate(range(0, n_out, CSR_CHUNK_ROWS)):
        rows = selected_rows[start:start+CSR_CHUNK_ROWS]

        starts = indptr[rows]
        ends = indptr[rows + 1]
        lens = (ends - starts)

        parts_data = []
        parts_ind = []
        for s, e in zip(starts, ends):
            if e > s:
                parts_data.append(data_in[s:e])
                parts_ind.append(ind_in[s:e])

        if parts_data:
            cat_data = np.concatenate(parts_data)
            cat_ind = np.concatenate(parts_ind)

            new_n = nnz_total + len(cat_data)
            d_data.resize((new_n,))
            d_ind.resize((new_n,))
            d_data[nnz_total:new_n] = cat_data
            d_ind[nnz_total:new_n] = cat_ind
            nnz_total = new_n

        csum = np.cumsum(lens)
        base = d_ptr[out_row]
        for k, v in enumerate(csum, start=1):
            d_ptr[out_row + k] = base + v

        out_row += len(rows)

        if chunk_i % 5 == 0:
            print(f"[csr] rows {out_row}/{n_out}")

    if d_ptr[-1] != nnz_total:
        raise RuntimeError("CSR integrity fail")

    print("[csr] done | nnz:", nnz_total)

# -----------------------
# BUILD LOCAL FILE
# -----------------------

if os.path.exists(LOCAL_DST):
    os.remove(LOCAL_DST)

with h5py.File(SRC_H5AD, "r") as fin, h5py.File(LOCAL_DST, "w") as fout:

    fout.attrs["encoding-type"] = "anndata"
    fout.attrs["encoding-version"] = "0.1.0"

    # ---- OBS ----
    obs_in = fin["obs"]

    obs_index = np.arange(n_sel).astype("U")  # NEW index
    obs_cols = {}

    print("[obs] reading columns")
    for c in KEEP_OBS_COLS:
        if c in obs_in:
            print("   ->", c)
            obs_cols[c] = read_obs_col(obs_in, c, selected_pos)

    # add source row for integrity
    obs_cols["_src_row"] = selected_pos

    write_dataframe_group(fout, "obs", obs_index, obs_cols)

    # ---- VAR (raw) ----
    var_src = fin["raw"]["var"]
    var_index = _as_str(var_src["_index"][()])

    var_cols = {}
    for c in KEEP_RAW_VAR_COLS:
        if c in var_src:
            # FIX: dataset vs group (categorical)
            var_cols[c] = read_var_col(var_src, c)

    write_dataframe_group(fout, "var", var_index, var_cols)

    # ---- X (raw csr subset) ----
    csr_subset(fin["raw"]["X"], fout, selected_pos, len(var_index))

print("[ok] local file written:", LOCAL_DST)

# -----------------------
# COPY TO DRIVE
# -----------------------

os.makedirs(OUT_DIR_DRIVE, exist_ok=True)

tmp_drive = DRIVE_DST + ".tmp"
if os.path.exists(tmp_drive):
    os.remove(tmp_drive)

print("[copy] local -> drive")
shutil.copy2(LOCAL_DST, tmp_drive)

if os.path.exists(DRIVE_DST):
    os.remove(DRIVE_DST)

os.rename(tmp_drive, DRIVE_DST)

print("[ok] drive file ready:", DRIVE_DST)


[sel] n_cells: 914746
H5 __getitem__ got: '/' type: <class 'str'>
H5 __getitem__ got: '/' type: <class 'str'>
H5 __getitem__ got: 'obs' type: <class 'str'>
[obs] reading columns
   -> sampleID
H5 __getitem__ got: 'sampleID' type: <class 'str'>
H5 __getitem__ got: 'categories' type: <class 'str'>
H5 __getitem__ got: 'codes' type: <class 'str'>
   -> donor_id
H5 __getitem__ got: 'donor_id' type: <class 'str'>
H5 __getitem__ got: 'categories' type: <class 'str'>
H5 __getitem__ got: 'codes' type: <class 'str'>
   -> CoVID-19 severity
H5 __getitem__ got: 'CoVID-19 severity' type: <class 'str'>
H5 __getitem__ got: 'categories' type: <class 'str'>
H5 __getitem__ got: 'codes' type: <class 'str'>
   -> celltype
H5 __getitem__ got: 'celltype' type: <class 'str'>
H5 __getitem__ got: 'categories' type: <class 'str'>
H5 __getitem__ got: 'codes' type: <class 'str'>
   -> majorType
H5 __getitem__ got: 'majorType' type: <class 'str'>
H5 __getitem__ got: 'categories' type: <class 'str'>
H5 __getitem__ 

In [7]:
!pip install anndata

In [12]:
import anndata as ad
import numpy as np

p = DRIVE_DST
adata = ad.read_h5ad(p, backed="r")
print("X:", adata.X.shape, type(adata.X))
print("obs:", adata.obs.shape, "cols:", list(adata.obs.columns)[:10])
print("var:", adata.var.shape, "cols:", list(adata.var.columns)[:10])
print("X nnz (approx):", adata.X.nnz if hasattr(adata.X, "nnz") else "NA")


H5 __getitem__ got: '/' type: <class 'str'>
H5 __getitem__ got: 'obs' type: <class 'str'>
H5 __getitem__ got: 'sampleID' type: <class 'str'>
H5 __getitem__ got: 'donor_id' type: <class 'str'>
H5 __getitem__ got: 'CoVID-19 severity' type: <class 'str'>
H5 __getitem__ got: 'celltype' type: <class 'str'>
H5 __getitem__ got: 'majorType' type: <class 'str'>
H5 __getitem__ got: 'City' type: <class 'str'>
H5 __getitem__ got: '_src_row' type: <class 'str'>
H5 __getitem__ got: '_index' type: <class 'str'>
H5 __getitem__ got: 'var' type: <class 'str'>
H5 __getitem__ got: 'feature_biotype' type: <class 'str'>
H5 __getitem__ got: 'feature_length' type: <class 'str'>
H5 __getitem__ got: 'feature_name' type: <class 'str'>
H5 __getitem__ got: 'feature_reference' type: <class 'str'>
H5 __getitem__ got: 'feature_type' type: <class 'str'>
H5 __getitem__ got: '_index' type: <class 'str'>
H5 __getitem__ got: 'obs' type: <class 'str'>
H5 __getitem__ got: 'X' type: <class 'str'>
H5 __getitem__ got: 'X' type

In [1]:
import numpy as np
import h5py

SRC_H5AD = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/Ren_et_al_COVID_2021.h5ad"
DST_H5AD = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts.h5ad"
SEL_NPY  = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/ren_cap4096_min100_seed0_cellpos.npy"

def scan_indptr_monotone(indptr_ds, chunk=2_000_000):
    n = indptr_ds.shape[0]
    prev_last = None
    for start in range(0, n, chunk):
        end = min(n, start + chunk)
        ip = indptr_ds[start:end]  # slice, not full
        if ip.size == 0:
            continue
        if start == 0:
            assert int(ip[0]) == 0, "indptr[0] != 0"
        else:
            # boundary monotonicity
            assert int(ip[0]) >= int(prev_last), "indptr not nondecreasing at chunk boundary"
        # within-chunk monotonicity
        assert np.all(ip[1:] >= ip[:-1]), "indptr not nondecreasing within chunk"
        prev_last = ip[-1]
    nnz = int(indptr_ds[-1])  # only last element
    return nnz

def scan_indices_bounds(indices_ds, n_vars, chunk=5_000_000):
    n = indices_ds.shape[0]
    mn = None
    mx = None
    for start in range(0, n, chunk):
        end = min(n, start + chunk)
        x = indices_ds[start:end]
        if x.size == 0:
            continue
        lo = int(x.min(initial=0))
        hi = int(x.max(initial=0))
        mn = lo if mn is None else min(mn, lo)
        mx = hi if mx is None else max(mx, hi)
    assert mn is not None, "indices empty?"
    assert mn >= 0, f"indices has negative (min={mn})"
    assert mx < n_vars, f"indices out of range (max={mx}, n_vars={n_vars})"
    return mn, mx

def csr_row_sparse_from_h5(Xg, row):
    # read only indptr[row:row+2]
    ip = Xg["indptr"][row:row+2]
    s = int(ip[0]); e = int(ip[1])
    cols = Xg["indices"][s:e]
    vals = Xg["data"][s:e]
    if cols.size == 0:
        return cols, vals
    o = np.argsort(cols)
    return cols[o], vals[o]

def compare_rows_sparse(srcXg, dstXg, src_row, dst_row):
    s_cols, s_vals = csr_row_sparse_from_h5(srcXg, src_row)
    d_cols, d_vals = csr_row_sparse_from_h5(dstXg, dst_row)

    if s_cols.size != d_cols.size:
        return False, f"nnz mismatch src={s_cols.size} dst={d_cols.size}"
    if s_cols.size == 0:
        return True, "both empty"
    if not np.array_equal(s_cols, d_cols):
        return False, "col mismatch"

    # values: strict
    if s_vals.dtype.kind in ("i","u") and d_vals.dtype.kind in ("i","u"):
        ok = np.array_equal(s_vals, d_vals)
        return ok, "int equal" if ok else "int value mismatch"
    ok = np.allclose(s_vals, d_vals, rtol=0, atol=0)
    return ok, "float equal" if ok else "float value mismatch"

# -----------------------
# Load selection (this is ~7MB, OK)
# -----------------------
sel = np.load(SEL_NPY).astype(np.int64)
sel.sort()

# -----------------------
# (A) HDF5 structure + obs/var index + _src_row == sel
# (B) CSR internal integrity without full indices load
# -----------------------
with h5py.File(DST_H5AD, "r") as fd:
    assert fd.attrs.get("encoding-type") == "anndata", "root encoding-type not anndata"
    assert "obs" in fd and "var" in fd and "X" in fd, "missing obs/var/X"

    for k in ["obs", "var"]:
        g = fd[k]
        assert g.attrs.get("encoding-type") == "dataframe", f"{k} not dataframe"
        assert "_index" in g.attrs, f"{k} missing attrs['_index']"
        idx_name = g.attrs["_index"]
        assert idx_name in g, f"{k} attrs['_index'] points to missing dataset: {idx_name}"
        assert "column-order" in g.attrs, f"{k} missing column-order"

    n_obs = fd["obs"][fd["obs"].attrs["_index"]].shape[0]
    n_vars = fd["var"][fd["var"].attrs["_index"]].shape[0]

    assert "_src_row" in fd["obs"], "obs missing _src_row"
    src_row = fd["obs"]["_src_row"][()]
    assert src_row.shape[0] == n_obs, "_src_row length mismatch"
    assert np.array_equal(np.asarray(src_row, dtype=np.int64), sel), "_src_row != sel.npy (sorted)"
    del src_row  # free

    Xg = fd["X"]
    assert Xg.attrs.get("encoding-type") == "csr_matrix", "X not csr_matrix"
    shape = tuple(np.asarray(Xg.attrs["shape"]).tolist())
    assert shape == (n_obs, n_vars), f"X shape attr mismatch: {shape} vs ({n_obs},{n_vars})"
    assert Xg["data"].shape == Xg["indices"].shape, "data/indices length mismatch"
    assert Xg["indptr"].shape == (n_obs + 1,), "indptr shape mismatch"

    # indptr monotonicity in chunks (cheap)
    nnz = scan_indptr_monotone(Xg["indptr"], chunk=2_000_000)
    assert nnz == Xg["data"].shape[0], f"nnz mismatch indptr[-1]={nnz} vs len(data)={Xg['data'].shape[0]}"

    # indices bounds in chunks (no full load)
    mn, mx = scan_indices_bounds(Xg["indices"], n_vars, chunk=5_000_000)

    print(f"[OK] dst csr integrity | shape={shape} nnz={nnz} indices_min={mn} indices_max={mx}")

# -----------------------
# (C) sampled equivalence vs source raw/X (no full loads)
# -----------------------
rng = np.random.default_rng(0)
n_check = 50
dst_rows = rng.choice(len(sel), size=min(n_check, len(sel)), replace=False)
src_rows = sel[dst_rows]

with h5py.File(SRC_H5AD, "r") as fs, h5py.File(DST_H5AD, "r") as fd:
    sX = fs["raw"]["X"]
    dX = fd["X"]

    for i, (dr, sr) in enumerate(zip(dst_rows, src_rows), start=1):
        ok, msg = compare_rows_sparse(sX, dX, int(sr), int(dr))
        if not ok:
            raise AssertionError(f"[FAIL] row {i}: dst_row={dr} src_row={sr} :: {msg}")

print(f"[OK] sampled raw/X equivalence passed (n_check={len(dst_rows)})")
print("[ALL OK] integrity checks passed (memory-safe)")


[OK] dst csr integrity | shape=(914746, 27131) nnz=1490211184 indices_min=0 indices_max=27130
[OK] sampled raw/X equivalence passed (n_check=50)
[ALL OK] integrity checks passed (memory-safe)


In [1]:
import numpy as np
import h5py

SRC_H5AD = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/Ren_et_al_COVID_2021.h5ad"
DST_H5AD = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts.h5ad"

def row_sum_from_h5_csr(Xg, row):
    ip = Xg["indptr"][row:row+2]
    s = int(ip[0]); e = int(ip[1])
    if e <= s:
        return 0.0, 0
    vals = Xg["data"][s:e]
    return float(np.sum(vals)), int(e - s)

def fast_block_value_checks(Xg, blocks=20, block_len=50_000, seed=0):
    rng = np.random.default_rng(seed)
    n = int(Xg["data"].shape[0])
    # sample a few contiguous blocks (much faster than random fancy indexing)
    mins = []
    maxs = []
    neg = False
    nonint_cnt = 0
    total = 0

    for _ in range(blocks):
        if n <= block_len:
            start = 0
        else:
            start = int(rng.integers(0, n - block_len))
        v = Xg["data"][start:start+block_len]

        mins.append(float(v.min(initial=0)))
        maxs.append(float(v.max(initial=0)))
        if np.any(v < 0):
            neg = True

        if not np.issubdtype(v.dtype, np.integer):
            nonint_cnt += int(np.sum(v != np.round(v)))
            total += int(v.size)
        else:
            total += int(v.size)

    frac_nonint = 0.0 if total == 0 else nonint_cnt / total
    return {
        "dtype": str(Xg["data"].dtype),
        "blocks": blocks,
        "block_len": block_len,
        "min": float(np.min(mins)) if mins else 0.0,
        "max": float(np.max(maxs)) if maxs else 0.0,
        "has_neg": bool(neg),
        "frac_nonint": float(frac_nonint),
    }

with h5py.File(SRC_H5AD, "r") as fs, h5py.File(DST_H5AD, "r") as fd:
    sX = fs["raw"]["X"]
    dX = fd["X"]

    print("[SRC raw/X] data dtype:", sX["data"].dtype)
    print("[DST X   ] data dtype:", dX["data"].dtype)

    src_stats = fast_block_value_checks(sX, blocks=20, block_len=50_000, seed=0)
    dst_stats = fast_block_value_checks(dX, blocks=20, block_len=50_000, seed=0)
    print("\n[Fast block value checks]")
    print("SRC:", src_stats)
    print("DST:", dst_stats)

    # row-sum/nnz match with fewer rows
    rng = np.random.default_rng(1)
    n_obs = int(dX["indptr"].shape[0] - 1)
    rows = rng.choice(n_obs, size=50, replace=False)

    mism = 0
    for r in rows:
        src_r = int(fd["obs"]["_src_row"][r])
        ss, snnz = row_sum_from_h5_csr(sX, src_r)
        ds, dnnz = row_sum_from_h5_csr(dX, r)
        if ss != ds or snnz != dnnz:
            mism += 1
            if mism <= 5:
                print(f"[mismatch] dst_row={r} src_row={src_r} sum_src={ss} sum_dst={ds} nnz_src={snnz} nnz_dst={dnnz}")

    assert mism == 0, f"row-sum/nnz mismatches: {mism}/50"
    print("\n[OK] fast rawness checks passed")


[SRC raw/X] data dtype: float32
[DST X   ] data dtype: float32

[Fast block value checks]
SRC: {'dtype': 'float32', 'blocks': 20, 'block_len': 50000, 'min': 0.0, 'max': 9912.0, 'has_neg': False, 'frac_nonint': 0.0}
DST: {'dtype': 'float32', 'blocks': 20, 'block_len': 50000, 'min': 0.0, 'max': 9400.0, 'has_neg': False, 'frac_nonint': 0.0}

[OK] fast rawness checks passed


In [5]:
import anndata as ad

DST_H5AD = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts.h5ad"
adata = ad.read_h5ad(DST_H5AD, backed = 'r')
print("majorType n_unique:", adata.obs["majorType"].nunique())
print("celltype  n_unique:", adata.obs["celltype"].nunique())
print(adata.obs["majorType"].value_counts().head())
print(adata.obs["celltype"].value_counts().head())


majorType n_unique: 7
celltype  n_unique: 6
majorType
C    397612
B    255549
M    198309
N     40828
D      8653
Name: count, dtype: int64
celltype
T    397612
B    264167
M    198309
N     40828
D      8653
Name: count, dtype: int64


In [7]:
!pip install scikit-misc --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 4.7 MB/s eta 0:00:00


In [8]:
import numpy as np
import scanpy as sc
import anndata as ad

DST_H5AD = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts.h5ad"
OUT_DIR  = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021"

HVG_N = 4000
TARGET_CELLS = 200_000
BATCH_KEY = "sampleID"

HVG_TXT = f"{OUT_DIR}/hvg_seuratv3_top{HVG_N}.txt"
HVG_NPY = f"{OUT_DIR}/hvg_seuratv3_top{HVG_N}.npy"

rng = np.random.default_rng(0)

# 1️⃣ backed로 열기
adata = ad.read_h5ad(DST_H5AD, backed="r")

# 2️⃣ 랜덤 200k 셀 선택
n = adata.n_obs
sel = rng.choice(n, size=min(TARGET_CELLS, n), replace=False)
sel.sort()

print(f"[sample] {len(sel)} / {n}")

# 3️⃣ 선택 셀만 메모리로 로드
adata_fit = adata[sel, :].to_memory()

print("[memory] shape:", adata_fit.shape)

# 4️⃣ HVG 계산 (raw counts 기준)
sc.pp.highly_variable_genes(
    adata_fit,
    n_top_genes=HVG_N,
    flavor="seurat_v3",
    batch_key=BATCH_KEY,
    subset=False
)

hvg_mask = adata_fit.var["highly_variable"].to_numpy()
hvg_genes = adata_fit.var_names[hvg_mask].to_numpy()

print("[HVG] selected:", hvg_genes.size)

# 5️⃣ 저장
np.save(HVG_NPY, hvg_genes)

with open(HVG_TXT, "w") as f:
    for g in hvg_genes:
        f.write(g + "\n")

print("[OK] saved:", HVG_NPY)


[sample] 200000 / 914746
[memory] shape: (200000, 27131)
[HVG] selected: 4000
[OK] saved: /content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/hvg_seuratv3_top4000.npy


In [10]:
import os
import numpy as np
import h5py

DST_H5AD = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts.h5ad"
OUT_H5AD = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts_hvg4000.h5ad"

HVG_NPY  = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/hvg_seuratv3_top4000.npy"
HVG_TXT = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/hvg_seuratv3_top4000.txt"

CSR_CHUNK_ROWS = 32768

# -----------------------
# Helpers for dataframe groups
# -----------------------
def load_hvg_list(hvg_npy, fallback_txt=None):
    # 1) try npy without pickle
    try:
        x = np.load(hvg_npy, allow_pickle=False)
        return np.asarray(x, dtype=str)
    except ValueError as e:
        # object array -> require pickle
        x = np.load(hvg_npy, allow_pickle=True)
        x = np.asarray(x, dtype=str)
        return x
    except FileNotFoundError:
        pass

    if fallback_txt is None:
        raise FileNotFoundError(f"missing HVG npy: {hvg_npy} (and no txt fallback)")
    # 2) txt fallback
    with open(fallback_txt, "r") as f:
        genes = [line.strip() for line in f if line.strip()]
    return np.asarray(genes, dtype=str)

def _as_str(arr):
    arr = np.asarray(arr)
    if arr.dtype.kind == "U":
        return arr
    if arr.dtype.kind in ("S", "O"):
        return arr.astype("U")
    return arr

def write_dataframe_group(fout, key, index_vals, cols_dict):
    g = fout.create_group(key)
    g.attrs["encoding-type"] = "dataframe"
    g.attrs["encoding-version"] = "0.2.0"
    g.attrs["_index"] = "_index"

    str_dt = h5py.string_dtype("utf-8")

    g.create_dataset("_index", data=_as_str(index_vals).astype(object), dtype=str_dt)

    col_order = []
    for c, arr in cols_dict.items():
        col_order.append(c)
        a = np.asarray(arr)
        if a.dtype.kind in ("U", "S", "O"):
            g.create_dataset(c, data=_as_str(a).astype(object), dtype=str_dt)
        else:
            g.create_dataset(c, data=a, dtype=a.dtype)

    g.attrs["column-order"] = np.asarray(col_order).astype(object)

def read_dataframe_group(fin, key):
    g = fin[key]
    assert g.attrs.get("encoding-type") == "dataframe"
    idx_name = g.attrs["_index"]
    index_vals = g[idx_name][()]
    col_order = _as_str(g.attrs["column-order"])
    cols = {}
    for c in col_order:
        c = str(c)
        cols[c] = g[c][()]
        if np.asarray(cols[c]).dtype.kind in ("S", "O"):
            cols[c] = _as_str(cols[c])
    return index_vals, col_order, cols

# -----------------------
# CSR: column-subset streaming (CSR -> CSR)
# -----------------------
def csr_col_subset(Xin, fout, keep_mask, old2new, n_vars_out, chunk_rows=32768):
    data_in = Xin["data"]
    ind_in  = Xin["indices"]
    ptr_in  = Xin["indptr"]

    indptr = ptr_in[()]  # ~ (n_obs+1) ints; OK memory
    n_obs = indptr.shape[0] - 1

    Xout = fout.create_group("X")
    Xout.attrs["encoding-type"] = "csr_matrix"
    Xout.attrs["encoding-version"] = "0.1.0"
    Xout.attrs["shape"] = np.array([n_obs, n_vars_out], dtype=np.int64)

    d_data = Xout.create_dataset("data", shape=(0,), maxshape=(None,), dtype=data_in.dtype, chunks=True)
    d_ind  = Xout.create_dataset("indices", shape=(0,), maxshape=(None,), dtype=ind_in.dtype, chunks=True)
    d_ptr  = Xout.create_dataset("indptr", shape=(n_obs + 1,), dtype=ptr_in.dtype)

    d_ptr[0] = 0
    nnz_total = 0
    out_row = 0

    print("[csr-colsub] start | n_obs:", n_obs, "n_vars_out:", n_vars_out)

    for chunk_i, start in enumerate(range(0, n_obs, chunk_rows)):
        end = min(n_obs, start + chunk_rows)

        # We'll build this chunk row-by-row (streaming)
        ptr_chunk = np.empty((end - start + 1,), dtype=d_ptr.dtype)
        ptr_chunk[0] = d_ptr[out_row]
        chunk_data_parts = []
        chunk_ind_parts = []
        nnz_in_chunk = 0

        for r in range(start, end):
            s = int(indptr[r]); e = int(indptr[r+1])
            if e <= s:
                ptr_chunk[r - start + 1] = ptr_chunk[r - start]
                continue

            cols = ind_in[s:e]
            vals = data_in[s:e]

            # filter columns
            km = keep_mask[cols]
            if np.any(km):
                cols2 = cols[km]
                vals2 = vals[km]

                # remap old col idx -> new col idx
                new_cols = old2new[cols2]
                # old2new should never be -1 after masking, but guard anyway
                good = new_cols >= 0
                if np.any(~good):
                    new_cols = new_cols[good]
                    vals2 = vals2[good]

                chunk_ind_parts.append(new_cols.astype(ind_in.dtype, copy=False))
                chunk_data_parts.append(vals2.astype(data_in.dtype, copy=False))
                nnz_in_row = int(new_cols.size)
            else:
                nnz_in_row = 0

            nnz_in_chunk += nnz_in_row
            ptr_chunk[r - start + 1] = ptr_chunk[r - start] + nnz_in_row

        # append chunk to datasets
        if nnz_in_chunk > 0:
            cat_data = np.concatenate(chunk_data_parts) if chunk_data_parts else np.empty((0,), dtype=data_in.dtype)
            cat_ind  = np.concatenate(chunk_ind_parts)  if chunk_ind_parts  else np.empty((0,), dtype=ind_in.dtype)

            new_n = nnz_total + nnz_in_chunk
            d_data.resize((new_n,))
            d_ind.resize((new_n,))
            d_data[nnz_total:new_n] = cat_data
            d_ind[nnz_total:new_n]  = cat_ind
            nnz_total = new_n

        # write indptr chunk
        d_ptr[out_row:out_row + (end - start) + 1] = ptr_chunk
        out_row += (end - start)

        if chunk_i % 5 == 0:
            print(f"[csr-colsub] rows {out_row}/{n_obs} | nnz={nnz_total}")

    if int(d_ptr[-1]) != int(nnz_total):
        raise RuntimeError(f"CSR integrity fail: indptr[-1]={int(d_ptr[-1])} nnz={nnz_total}")

    print("[csr-colsub] done | nnz:", nnz_total)

# -----------------------
# Main
# -----------------------
hvg = load_hvg_list(HVG_NPY, fallback_txt=HVG_TXT)
hvg_set = set(hvg.tolist())
assert hvg.ndim == 1 and len(hvg) == 4000, f"bad HVG list shape/len: {hvg.shape} {len(hvg)}"

if os.path.exists(OUT_H5AD):
    os.remove(OUT_H5AD)

with h5py.File(DST_H5AD, "r") as fin, h5py.File(OUT_H5AD, "w") as fout:
    # root attrs
    fout.attrs["encoding-type"] = "anndata"
    fout.attrs["encoding-version"] = "0.1.0"

    # obs: copy as-is (already minimal and valid)
    obs_index, obs_col_order, obs_cols = read_dataframe_group(fin, "obs")
    write_dataframe_group(fout, "obs", obs_index, obs_cols)

    # var: subset by HVG list
    var_index_full, var_col_order, var_cols_full = read_dataframe_group(fin, "var")
    var_names_full = _as_str(var_index_full).astype(str)

    keep = np.array([g in hvg_set for g in var_names_full], dtype=bool)
    n_keep = int(keep.sum())
    print("[var] keep:", n_keep, "/", len(var_names_full))
    assert n_keep == len(hvg), f"HVG list size={len(hvg)} but matched var={n_keep} (name mismatch?)"

    var_index = var_index_full[keep]
    var_cols = {c: np.asarray(var_cols_full[c])[keep] for c in var_col_order}
    write_dataframe_group(fout, "var", var_index, var_cols)

    # build old->new mapping
    old2new = np.full((len(var_names_full),), -1, dtype=np.int64)
    old2new[np.flatnonzero(keep)] = np.arange(n_keep, dtype=np.int64)

    # X: column-subset CSR (streaming)
    keep_mask = keep  # bool length n_vars_full
    csr_col_subset(fin["X"], fout, keep_mask=keep_mask, old2new=old2new, n_vars_out=n_keep, chunk_rows=CSR_CHUNK_ROWS)

print("[OK] wrote HVG-only h5ad:", OUT_H5AD)


[var] keep: 4000 / 27131
[csr-colsub] start | n_obs: 914746 n_vars_out: 4000
[csr-colsub] rows 32768/914746 | nnz=6901464
[csr-colsub] rows 196608/914746 | nnz=41082920
[csr-colsub] rows 360448/914746 | nnz=75008019
[csr-colsub] rows 524288/914746 | nnz=99421327
[csr-colsub] rows 688128/914746 | nnz=133907524
[csr-colsub] rows 851968/914746 | nnz=163534034
[csr-colsub] done | nnz: 174538301
[OK] wrote HVG-only h5ad: /content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts_hvg4000.h5ad


In [11]:
import numpy as np
import h5py

HVG_H5AD = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts_hvg4000.h5ad"

def _as_u(x):
    x = np.asarray(x)
    if x.dtype.kind in ("S","O"):
        return x.astype("U")
    return x

with h5py.File(HVG_H5AD, "r") as f:
    # root
    assert f.attrs.get("encoding-type") == "anndata", "root encoding-type != anndata"
    assert "obs" in f and "var" in f and "X" in f, "missing obs/var/X"

    # dataframe spec
    for k in ["obs","var"]:
        g = f[k]
        assert g.attrs.get("encoding-type") == "dataframe", f"{k}: encoding-type != dataframe"
        assert "_index" in g.attrs, f"{k}: missing attrs['_index']"
        idx_name = g.attrs["_index"]
        assert idx_name in g, f"{k}: attrs['_index'] points to missing dataset '{idx_name}'"
        assert "column-order" in g.attrs, f"{k}: missing attrs['column-order']"
        col_order = _as_u(g.attrs["column-order"])
        for c in col_order:
            c = str(c)
            assert c in g, f"{k}: column '{c}' missing dataset"
        # basic lengths
        n = g[idx_name].shape[0]
        for c in col_order:
            assert g[str(c)].shape[0] == n, f"{k}: column {c} length mismatch"

    n_obs = f["obs"][f["obs"].attrs["_index"]].shape[0]
    n_vars = f["var"][f["var"].attrs["_index"]].shape[0]

    # X csr spec
    X = f["X"]
    assert X.attrs.get("encoding-type") == "csr_matrix", "X: encoding-type != csr_matrix"
    assert "data" in X and "indices" in X and "indptr" in X, "X: missing data/indices/indptr"
    shape = tuple(np.asarray(X.attrs["shape"]).tolist())
    assert shape == (n_obs, n_vars), f"X shape attr {shape} != (n_obs,n_vars)=({n_obs},{n_vars})"

    assert X["indptr"].shape == (n_obs + 1,), "X: indptr shape mismatch"
    assert X["data"].shape == X["indices"].shape, "X: data/indices length mismatch"
    # cheap nnz check
    nnz = int(X["indptr"][-1])
    assert nnz == X["data"].shape[0], "X: nnz mismatch indptr[-1] vs len(data)"

print("[OK] h5ad spec-level checks passed")


[OK] h5ad spec-level checks passed


In [12]:
import anndata as ad
import os

HVG_H5AD = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts_hvg4000.h5ad"
TMP_RT   = "/content/adata_hvg4000_roundtrip_tmp.h5ad"

adata = ad.read_h5ad(HVG_H5AD, backed="r")
print("[OK] anndata open:", adata.shape, type(adata.X))

# minimal sanity
assert adata.n_obs > 0 and adata.n_vars == 4000, "unexpected shape"
assert "sampleID" in adata.obs.columns, "missing sampleID in obs"
assert adata.X.shape == (adata.n_obs, adata.n_vars), "X shape mismatch"

# round-trip: write a small copy (backed -> to_memory -> write)
# NOTE: this loads nothing huge; it just forces anndata to serialize its view of the object.
adata_mem = adata[:1000, :200].to_memory()
if os.path.exists(TMP_RT):
    os.remove(TMP_RT)
adata_mem.write_h5ad(TMP_RT)

# re-open written file
adata2 = ad.read_h5ad(TMP_RT, backed="r")
print("[OK] round-trip open:", adata2.shape)

print("[OK] anndata compatibility passed")


[OK] anndata open: (914746, 4000) <class 'anndata._core.sparse_dataset._CSRDataset'>
[OK] round-trip open: (1000, 200)
[OK] anndata compatibility passed


In [2]:
!pip install scvi-tools --quiet

In [3]:
!pip uninstall -y scvi

In [1]:
import anndata as ad
import numpy as np

HVG_H5AD = "/content/drive/MyDrive/scMILD_residual/data/Ren_COVID_2021/adata_cap4096_min100_milmin_rawcounts_hvg4000.h5ad"

adata = ad.read_h5ad(HVG_H5AD, backed="r")

# tiny subset (메타/포맷 검증용)
rng = np.random.default_rng(0)
cells = rng.choice(adata.n_obs, size=5000, replace=False)
cells.sort()

adata_small = adata[cells, :].to_memory()

# scanpy: normalize/log 체크는 필요 없지만, 최소한 HVG flag나 PCA 같은 게 되는지 확인할 수 있음
import scanpy as sc
sc.pp.normalize_total(adata_small, target_sum=1e4)  # optional; if you plan scVI on raw, you can skip normalization entirely
# (너는 scVI raw로 갈 거니까 여기선 그냥 스모크로만)

# scVI smoke
import scvi
scvi.model.SCVI.setup_anndata(adata_small, batch_key="sampleID")
m = scvi.model.SCVI(adata_small, n_latent=16)

# 1 epoch만: 포맷/데이터 로더 깨지는지 확인용
m.train(max_epochs=1, check_val_every_n_epoch=1, enable_progress_bar=True)
z = m.get_latent_representation()
print("[OK] scVI latent:", z.shape)


/usr/local/lib/python3.12/dist-packages/scvi/data/fields/_base_field.py:63: UserWarning: adata.X does not contain unnormalized count data. Are you sure this is what you want?
  self.validate_field(adata)
INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Training:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)
/usr/local/lib/python3.12/dist-packages/scvi/module/_vae.py:573: UserWarning: The value argument must be within the support of the distribution
  reconst_loss = -generative_outputs[MODULE_KEYS.PX_KEY].log_prob(x).sum(-1)
INFO: `Trainer.fit` stopped: `max_epochs=1` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=1` reached.


[OK] scVI latent: (5000, 16)
